# Gemma-3 (270M) Instagram Caption Fine-Tuning

Fine-tuned using the [newadays/alpaca_ig_post](https://huggingface.co/datasets/newadays/alpaca_ig_post) dataset from Hugging Face.

## How to Run

**Google Colab:**
1. Open this notebook in Colab
2. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
3. Click **Runtime → Run all**

**VS Code (Local):**
1. Install the Jupyter extension
2. Select a Python kernel with PyTorch and CUDA installed
3. Run `pip install unsloth` in a terminal
4. Run all cells sequentially

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastModel
import torch
max_seq_length = 2048
fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-270m-it",
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
from datasets import load_dataset
dataset = load_dataset("newadays/alpaca_ig_post", split="train")
dataset[0]

README.md:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

createkap_training_v2_alpaca.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1551 [00:00<?, ? examples/s]

{'instruction': 'Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n\n2 - Itemize the good features of a good property in a List\n\n3 - Include the location of the listing\n\n3 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n\n4 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n\n- Remember to use proper grammar throughout the caption\n\n- Keep the caption short and to the point. \n\n- Use an enthusiastic tone throughout the caption.\n\n- Create a Sense of Urgency throughout the caption.\n\n- Remember to itemize the features in a List\n\n```\n\na white house with the number 2919 on the front\n\n```',

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Thytu's ChessInstruct](https://huggingface.co/datasets/Thytu/ChessInstruct) dataset. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

We now use `convert_to_chatml` to try converting datasets to the correct format for finetuning purposes!

In [4]:
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": example["instruction"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]}
        ]
    }

dataset = dataset.map(
    convert_to_chatml
)

Map:   0%|          | 0/1551 [00:00<?, ? examples/s]

We now have to apply the chat template for `Gemma3` onto the conversations, and save it to `text`.

In [5]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/1551 [00:00<?, ? examples/s]

In [6]:
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


<a name="Train"></a>
### Train the model
Now let's train our model. We do 100 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [7]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1551 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [8]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=6):   0%|          | 0/1551 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/1551 [00:00<?, ? examples/s]

In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,551 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,593,984 of 275,692,160 (2.75% trained)


Step,Training Loss
1,4.557700
2,4.914400
3,4.712400
4,3.794700
5,3.997300
6,4.136500
7,3.443700
8,3.885300
9,3.841200
10,3.662100


Now let's print the masked out example - you should see only the answer is present:

In [10]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
2.307 GB of memory reserved.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
# Save merged model (LoRA weights folded into base, float16)
model.save_pretrained_merged(
    "gemma_3_ig_post_merged", tokenizer, save_method="merged_16bit",
)
model.push_to_hub_merged(
    "newadays/gemma_3_ig_post_merged", tokenizer,
    save_method="merged_16bit", token="YOUR_HF_TOKEN",
)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `gemma_3_ig_post_merged`: 100%|██████████| 1/1 [00:07<00:00,  7.91s/it]


Successfully copied all 1 files from cache to `gemma_3_ig_post_merged`
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `gemma_3_ig_post_merged`: 100%|██████████| 1/1 [00:00<00:00, 43.86it/s]


Successfully copied all 1 files from cache to `gemma_3_ig_post_merged`


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._merged/model.safetensors:   6%|5         | 31.9MB /  536MB            

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 11.67s/it]


Unsloth: Merge process complete. Saved to `/content/newadays/gemma_3_ig_post_merged`


Now if you want to load the merged model we just saved for inference, set `False` to `True`:

In [12]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "gemma_3_ig_post_merged", # YOUR MERGED MODEL
        max_seq_length = 2048,
        load_in_4bit = False,
    )

==((====))==  Unsloth 2026.3.4: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


### Inference Examples

Below are examples of generating Instagram captions using the fine-tuned model. Each example uses:
- **system message**: The full instruction prompt describing the task, brand, and guidelines
- **user message**: Left empty since the image context is already in the system prompt
- **`add_generation_prompt=True`**: Tells the tokenizer to append `<start_of_turn>model\n` so the model knows to generate a response
- **Sampling settings**: `temperature=1.0`, `top_p=0.95`, `top_k=64` (recommended by the Gemma-3 team)

#### Example 1 - "Just Listed" White House
Generates a caption for a newly listed white house property in Los Angeles.

In [13]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense of Urgency throughout the caption.\n- Remember to itemize the features in a List\n\n```\na white house with the words just listed above it\n```"},
    {"role": "user", "content": ""},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
).removeprefix('<bos>')

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 1, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

New listing in the neighborhood of Oakwood! 🏡

This 3-bedroom, 2-bath home is perfect for a family or couple.

*   Large backyard with a grassy area perfect for grilling or playing.

*   Large open concept living room with a cozy fireplace and large window overlooking the backyard.

*   Hallway with a slip-resistant slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floor.

*   Hallway with a slip-resistant tile floo

#### Example 2 - Luxury Pool Home
Generates a caption for a luxury property with a pool and palm trees.

In [14]:
messages = [
    {"role": "system", "content": "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense of Urgency throughout the caption.\n- Remember to itemize the features in a List\n\n```\na modern luxury home with a swimming pool and palm trees in the backyard\n```"},
    {"role": "user", "content": ""},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
).removeprefix('<bos>')

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 1, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The perfect coastal living experience for you! This 5,000 sqft property is a must-have for coastal living enthusiasts. Featuring a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage, this home is perfect for a wide range of uses.

The spacious main floor boasts a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage.

The spacious main floor features a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage.

The spacious main floor features a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage.

The spacious main floor features a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage.

The spacious main floor features a 6,500 sqft main floor with a 6,500 sqft kitchen, 2 bedrooms, 2 baths, and a 2,500 sqft garage.

The spacious main floor features a 6,500 sqft main floor with a 6,500 sqf

#### Example 3 - Penthouse with City Views
Generates a caption for a high-rise penthouse overlooking downtown Los Angeles.

In [15]:
messages = [
    {"role": "system", "content": "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense of Urgency throughout the caption.\n- Remember to itemize the features in a List\n\n```\na penthouse balcony with floor to ceiling windows overlooking the downtown city skyline at sunset\n```"},
    {"role": "user", "content": ""},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
).removeprefix('<bos>')

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 1, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Experience the magic ofমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানমহানম